# AI Warp Tool API

Using the proxy server endpoints to route requests to local or cloud backends.

In [2]:
import requests

WARP_URL = "http://localhost:11435"

def chat(model, messages, stream=False):
    r = requests.post(f"{WARP_URL}/api/chat", json={
        "model": model,
        "messages": messages,
        "stream": stream,
    }, stream=stream)
    print(r.status_code)
    if r.ok:
        if stream:
            for line in r.iter_lines():
                if line:
                    print(line)
        else:
            print(r.json()["message"]["content"])

## Version

In [3]:
r = requests.get(f"{WARP_URL}/api/version", params={"source": "local"})
print("Local:", r.status_code, r.text[:100])

r = requests.get(f"{WARP_URL}/api/version", params={"source": "cloud"})
print("Cloud:", r.status_code, r.text[:100])

Local: 200 {"version":"0.32.0"}
Cloud: 200 {"version":"0.0.0"}


## Tags

In [5]:
r = requests.get(f"{WARP_URL}/api/tags", params={"source": "local"})
print("Local:", r.status_code)
if r.ok:
    print([m["name"] for m in r.json().get("models", [])])

r = requests.get(f"{WARP_URL}/api/tags", params={"source": "cloud"})
print("Cloud:", r.status_code)
if r.ok:
    print([m["name"] for m in r.json().get("models", [])])

Local: 200
['gemma4:12b', 'gemma4:e4b', 'hf.co/jinaai/jina-embeddings-v5-omni-small-retrieval-GGUF:Q4_K_M', 'deepseek-v4-pro:cloud', 'gemma4:26b', 'nemotron-3-super:cloud', 'gpt-oss:120b-cloud', 'kimi-k2:1t-cloud', 'kimi-k2.5:cloud', 'kimi-k2.6:cloud', 'hf.co/mradermacher/InternVL2_5-4B-MPO-GGUF:Q8_0', 'hf.co/ngxson/Vintern-1B-v3_5-GGUF:Q8_0', 'qwen3.5:397b-cloud', 'gemma4:e2b', 'gemma4:31b-cloud', 'llama3.2:1b', 'gemma3:270m', 'gemma3:1b', 'gemma3:4b-cloud', 'embeddinggemma:300m']
Cloud: 405


## Chat — local model

In [45]:
chat("gemma4:e4b", [{"role": "user", "content": "hello"}])

200
Hello! How can I help you today? 😊


## Chat — cloud model via `-cloud` suffix

In [6]:
r = requests.post(f"{WARP_URL}/api/chat", json={
    "model": "gemma4:31b-cloud",
    "messages": [{"role": "system", "content": "You are a pirate. Answer every question like a pirate."},
                 {"role": "user", "content": "hello"}],
    "stream": False,
})
print(r.status_code)
if r.ok:
    print(r.json()["message"]["content"])

200
Ahoy there, matey! Avast! What brings a landlubber like ye to these salty waters? Speak up, or it's the plank for ye! 🏴‍☠️🦜


## Chat — cloud model via `?source=cloud` param

In [16]:
r = requests.post(f"{WARP_URL}/api/chat?source=cloud", json={
    "model": "gemma4:e4b",
    "messages": [{"role": "user", "content": "hello"}],
    "stream": False,
})
print(r.status_code)
if r.ok:
    print(r.json()["message"]["content"])

404


## Chat with system prompt

In [2]:
chat("gemma4:e4b", [
    {"role": "system", "content": "You are a pirate. Answer every question like a pirate."},
    {"role": "user", "content": "What is the capital of France?"},
])

200
Arrr, blast and barnacles! Ye askin' about the lands o' France, do ye? Listen close, landlubber, for this be simple enough even a swab can grasp it!

The capital port of call there be **Paris**, savvy? A fine city, I hear. Now get back to yer grog, afore I make ye walk the plank! 🏴‍☠️🦜


## Generate

In [3]:
r = requests.post(f"{WARP_URL}/api/generate", json={
    "model": "gemma4:31b-cloud",
    "messages": [
        {"role": "system", "content": "You are a pirate. Answer every question like a pirate."},
        {"role": "user", "content": "What is the capital of France?"}
    ],
    "stream": False,
})
print(r.status_code)
if r.ok:
    print(r.json()["response"])

200
Ahoy there, matey! That be Paris, the city o' lights! A fine place to plunder for art and fancy pastries, though I prefer the scent o' salt spray to the smell o' perfume! Savvy?


## Streaming chat

In [34]:
r = requests.post(f"{WARP_URL}/api/chat", json={
    "model": "gemma4:e4b",
    "messages": [{"role": "user", "content": "count to 3"}],
    "stream": True,
}, stream=True)
if r.ok:
    for line in r.iter_lines():
        if line:
            print(line)

b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.4326544Z","message":{"role":"assistant","content":"1"},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.5253613Z","message":{"role":"assistant","content":","},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.6142675Z","message":{"role":"assistant","content":" "},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.7073405Z","message":{"role":"assistant","content":"2"},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.8021142Z","message":{"role":"assistant","content":","},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.9167147Z","message":{"role":"assistant","content":" "},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:14.0296252Z","message":{"role":"assistant","content":"3"},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:14.1526181Z","message":{"role":"assistant","conte

## Embeddings

In [64]:
r = requests.post(f"{WARP_URL}/api/embed", json={
    "model": "embeddinggemma:300m",
    "input": "hello world",
})
print(r.status_code)
if r.ok:
    print(len(r.json()["embeddings"][0]), "dimensions")

200
768 dimensions


## OpenAI-compatible endpoint

Using `/v1/chat/completions` with OpenAI-compatible format. Routing works the same way (model suffix or `?source=` param).

In [ ]:
r = requests.post(f"{WARP_URL}/v1/chat/completions", json={
    "model": "gemma4:e4b",
    "messages": [{"role": "user", "content": "hello"}],
    "stream": False,
})
print(r.status_code)
if r.ok:
    data = r.json()
    print(data["choices"][0]["message"]["content"])

In [9]:
r = requests.post(f"{WARP_URL}/v1/chat/completions", json={
    "model": "gemma4:31b-cloud",
    "messages": [{"role": "system", "content": "You are a pirate."},
                 {"role": "user", "content": "hello"}],
    "stream": False,
})
print(r.status_code)
if r.ok:
    data = r.json()
    print(data["choices"][0]["message"]["content"])

200
Ahoy there, ye scurvy dog! 🏴‍☠️

Welcome aboard! Ye look like ye've come from a faraway shore. Speak plain: be ye seekin' buried treasure, a flagon o' rum, or be ye just lookin' to sail the seven seas with a crew of salty sea-dogs?

State yer business, or it's the plank for ye! Har har!


In [8]:
# Streaming via /v1/chat/completions (SSE format)
r = requests.post(f"{WARP_URL}/v1/chat/completions", json={
    "model": "gemma4:e4b",
    "messages": [{"role": "user", "content": "count to 3"}],
    "stream": True,
}, stream=True)
print(r.status_code)
if r.ok:
    for line in r.iter_lines():
        if line:
            decoded = line.decode() if isinstance(line, bytes) else line
            if decoded.startswith("data: ") and decoded != "data: [DONE]":
                import json
                chunk = json.loads(decoded[6:])
                content = chunk["choices"][0]["delta"].get("content", "")
                if content:
                    print(content, end="", flush=True)
    print()

200
1, 2, 3
